# 🧬 Panacée — Phase 1 : Pré-entraînement GNN (Kaggle)

**Phase 1** : Pré-entraînement auto-supervisé (Masked Graph Modeling) sur ~250k molécules ZINC.  
Produit `sovereign_encoder_v1.pth` — l'encodeur moléculaire de base pour la Phase 2.

### Chaîne complète :
```
Phase 1 (ce notebook) → sovereign_encoder_v1.pth
         ↓
Phase 2 (kaggle_phase2.ipynb) → best_toxicity_model.pth
         ↓
Phase 3 (kaggle_phase3.ipynb) → panacee_phase3_complete.pth
```

### Avant de lancer :
1. Dashboard local : `python -m webapp.run`
2. Tunnel : `ngrok http 8000`
3. Remplir `NGROK_URL` et `TOKEN` ci-dessous
4. ▶▶ Run All

> **GPU recommandé** (panneau droit → Accelerator : GPU T4).  
> Phase 1 sur 250k molécules : ~2-3h sur GPU, ~20h sur CPU.

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CONFIGURATION — modifier ces lignes ONLY    ║
# ╚══════════════════════════════════════════════╝

NGROK_URL     = "https://XXXX.ngrok-free.app"   # URL publique de votre dashboard
TOKEN         = "mon_token_secret_42"            # == PANACEE_INGEST_TOKEN dans .env
RUN_NAME      = "kaggle_phase1_run01"            # Nom affiché dans le dashboard
N_EPOCHS      = 100                             # GPU: 100, CPU: 20-30

# Nombre de molécules ZINC (None = tout, ~250k)
# Réduire pour un test rapide : 50000 → ~30min GPU
MAX_MOLECULES = None

print(f"Run         : {RUN_NAME}")
print(f"Epochs      : {N_EPOCHS}")
print(f"Molécules   : {MAX_MOLECULES or 'toutes (~250k)'}")
print(f"Dashboard   : {NGROK_URL}")

In [ ]:
# ── Vérification connexion dashboard ────────────────────────────────────────
import json
import urllib.request

try:
    r = urllib.request.urlopen(NGROK_URL.rstrip('/') + '/api/health', timeout=8)
    print(f"✅ Dashboard joignable : {json.loads(r.read())}")
except Exception as e:
    print(f"❌ Dashboard NON joignable ({e})")
    print("Checklist :")
    print("  1) python -m webapp.run         (terminal 1)")
    print("  2) ngrok http 8000              (terminal 2)")
    print("  3) NGROK_URL = URL affichée par ngrok")
    raise SystemExit("Corrigez avant de continuer.")

In [ ]:
# ── Variables d'environnement (AVANT tout import du projet) ─────────────────
import os

os.environ["PANACEE_PUSH_URL"]   = NGROK_URL
os.environ["PANACEE_PUSH_TOKEN"] = TOKEN
os.environ["PANACEE_PUSH_RUN"]   = RUN_NAME
os.environ["PYTHONIOENCODING"]   = "utf-8"

print("Variables d'environnement configurées ✓")

In [ ]:
# ── Installation des dépendances ────────────────────────────────────────────
import subprocess
import sys


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch-geometric", "rdkit")
print("Dépendances OK ✓")

In [ ]:
# ── Cloner le repo ──────────────────────────────────────────────────────────
from pathlib import Path

CLONE_DIR = Path("/kaggle/working/panacee")
REPO_URL  = "https://github.com/jumoras0000/savh_gnn.git"

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)
    print("Repo cloné ✓")
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)
    print("Repo mis à jour ✓")

PROJET = CLONE_DIR / "Projet_Panacee"
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))

print(f"Projet : {PROJET}")

In [ ]:
# ── Répertoire de sauvegarde ────────────────────────────────────────────────
# Toujours 'phase1' pour que Phase 2 trouve sovereign_encoder_v1.pth
SAVE_DIR = "/kaggle/working/checkpoints/phase1"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Checkpoints → {SAVE_DIR}")
print(f"  Encodeur final : {SAVE_DIR}/sovereign_encoder_v1.pth")

In [ ]:
# ── Lancer Phase 1 ──────────────────────────────────────────────────────────
import os as _os

_os.chdir(str(PROJET))

sys.argv = [
    "run_phase1.py",
    "--download",
    "--save_dir",  SAVE_DIR,
    "--epochs",    str(N_EPOCHS),
]
if MAX_MOLECULES:
    sys.argv += ["--max_molecules", str(MAX_MOLECULES)]

print(f"Démarrage Phase 1 — push → {os.environ['PANACEE_PUSH_URL']}")
print()

from run_phase1 import main

main()

In [ ]:
# ── Export du checkpoint → Output Kaggle ───────────────────────────────────
# Ce fichier sera disponible dans l'onglet Output pour être uploadé
# en Input du notebook Phase 2.
import shutil

OUTPUT_DIR = "/kaggle/output/panacee-phase1"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ckpt_src = f"{SAVE_DIR}/sovereign_encoder_v1.pth"
if Path(ckpt_src).exists():
    shutil.copy(ckpt_src, f"{OUTPUT_DIR}/sovereign_encoder_v1.pth")
    size_mb = Path(ckpt_src).stat().st_size / 1e6
    print(f"✅ Checkpoint exporté : {OUTPUT_DIR}/sovereign_encoder_v1.pth ({size_mb:.1f} MB)")
else:
    print(f"❌ Checkpoint introuvable : {ckpt_src}")
    print("   L'entraînement a peut-être échoué — vérifier les logs ci-dessus.")

print()
print("=" * 60)
print("ÉTAPE SUIVANTE : Phase 2")
print("  1. Onglet Output (droite) → télécharger sovereign_encoder_v1.pth")
print("  2. OU : Save Version → Output → 'Create dataset from output'")
print("  3. Uploader dans le notebook kaggle_phase2.ipynb")
print("="*60)